# 🌐 Division 4 — Soil & Weather API Ingestion Gateway
**Pipeline:** AI Precision Fertilizer System

**What happens here:**
1. Ingest geospatial soil baseline data via **SoilHive API** and **ISRIC SoilGrids API** using farmer GPS coordinates.
2. Query **Open-Meteo Weather API** for 72-hour precipitation, temperature, humidity, and soil moisture forecasts.
3. Apply regional agronomic logic (TNAU benchmarks) to fill nutrient availability ratings.
4. Standardize live API telemetry into the feature vector schema required by the ML Regressor (Division 3).
5. Export `soil_baseline.json` and `weather_telemetry.json` for live decision gating.

In [ ]:
# Cell 1 | Dependencies
# !pip install requests pandas numpy matplotlib

In [ ]:
# Cell 2 | Imports & Configuration
import requests
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Working directory
BASE_DIR = Path(r'D:\project sem 7\my-finalyear-project\UV\fertilizer')
print('Configured and ready.')

## 🛰️ 1. Soil Data Ingestion (SoilHive & SoilGrids)
We authenticate with SoilHive OAuth2 endpoints and query SoilGrids REST endpoints for depth-resolved chemical properties.

In [ ]:
# Cell 3 | SoilHive & SoilGrids Connectors
CLIENT_ID = '01a148ba3d3973ffa952edad4678d45c'
CLIENT_SECRET = '2LYGbjieTSoQn4cT5r2OK5SJOJngbhWB'

def get_soilhive_token(client_id, client_secret):
    """Generates a fresh OAuth2 token from auth.soilhive.ag"""
    auth_url = 'https://auth.soilhive.ag/oauth/token'
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'audience': 'https://api.soilhive.ag/'
    }
    headers = {'Content-Type': 'application/x-www-form-urlencoded'}
    try:
        res = requests.post(auth_url, data=payload, headers=headers, timeout=15)
        if res.status_code == 200:
            return res.json().get('access_token')
    except Exception as e:
        print(f'SoilHive Auth Error: {e}')
    return None

def query_soilhive(lat, lon, token):
    """Query SoilHive for physical baseline properties (Texture, SOC, Clay, Silt)."""
    if not token:
        return {}
    url = 'https://api.soilhive.ag/v1/soil-data-by-geometry'
    headers = {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json',
        'Accept': 'application/ld+json'
    }
    payload = {
        'geometry': f'POINT ({lon} {lat})',
        'datasets': [1, 20]  # GSOC & OpenLandMap
    }
    try:
        res = requests.post(url, json=payload, headers=headers, timeout=25)
        if res.status_code == 200:
            return res.json()
    except Exception as e:
        print(f'SoilHive query note: {e}')
    return {}

def query_soilgrids(lat, lon):
    """Query ISRIC SoilGrids 2.0 REST API for Nitrogen, pH, Organic Carbon, CEC."""
    url = 'https://rest.isric.org/soilgrids/v2.0/properties/query'
    params = {
        'lon': lon,
        'lat': lat,
        'property': ['nitrogen', 'phh2o', 'soc', 'cec']
    }
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        res = requests.get(url, params=params, headers=headers, timeout=25)
        if res.status_code == 200:
            return {l['name']: l for l in res.json().get('properties', {}).get('layers', [])}
    except Exception as e:
        print(f'SoilGrids query note: {e}')
    return {}

print('API connectors ready.')

In [ ]:
# Cell 4 | Fetch Soil Profile for Coordinates
def fetch_complete_soil_profile(lat, lon):
    token = get_soilhive_token(CLIENT_ID, CLIENT_SECRET)
    hive_data = query_soilhive(lat, lon, token)
    grids_data = query_soilgrids(lat, lon)
    
    # Default baselines for agricultural Tamil Nadu (Thanjavur delta standard)
    soil_texture = 'Loam'
    soc_stock = 25.4
    clay_pct = 30.0
    silt_pct = 25.0
    ph_val = 6.8
    nitrogen_cg_kg = 180.0  # ~1.8 g/kg
    cec_val = 22.5
    
    # Parse SoilHive
    for el in hive_data.get('dataFeedElement', []):
        items = {p.get('name'): p for p in el.get('item', [])}
        prop_val = items.get('Property', {}).get('value', '')
        depth = items.get('Depth', {}).get('value', '')
        conv = items.get('Conversion', {}).get('value')
        val = items.get('Value', {}).get('value')
        
        if 'usda_texture' in str(prop_val) and conv:
            soil_texture = conv
        if ('soc_0-30cm' in str(prop_val) or 'organic.carbon' in str(prop_val)) and isinstance(val, (int, float)):
            soc_stock = float(val)
        if 'sol_clay' in str(prop_val) and isinstance(val, (int, float)):
            clay_pct = float(val)
        if 'sol_silt' in str(prop_val) and isinstance(val, (int, float)):
            silt_pct = float(val)
            
    # Parse SoilGrids
    if 'phh2o' in grids_data:
        for d in grids_data['phh2o'].get('depths', []):
            m = d.get('values', {}).get('mean')
            if m is not None:
                ph_val = m / 10.0
                break
    if 'nitrogen' in grids_data:
        for d in grids_data['nitrogen'].get('depths', []):
            m = d.get('values', {}).get('mean')
            if m is not None:
                nitrogen_cg_kg = m
                break
    if 'cec' in grids_data:
        for d in grids_data['cec'].get('depths', []):
            m = d.get('values', {}).get('mean')
            if m is not None:
                cec_val = m / 10.0
                break
                
    # Derived nutrient mg/kg estimations for model input
    # (Nitrogen cg/kg -> /10 = g/kg -> *1000/20 for available fraction ~ 50-180 mg/kg)
    est_n_mg_kg = round(float(nitrogen_cg_kg * 0.45), 2)
    est_p_mg_kg = 35.5 if 'clay' in soil_texture.lower() else (22.0 if 'sand' in soil_texture.lower() else 42.0)
    est_k_mg_kg = 145.0 if 'clay' in soil_texture.lower() else (95.0 if 'sand' in soil_texture.lower() else 130.0)
    
    profile = {
        'Latitude': lat,
        'Longitude': lon,
        'Soil_Texture': soil_texture,
        'Soil_pH': round(ph_val, 2),
        'Soil_Organic_Matter_Percent': round(soc_stock * 0.1, 2),  # approximation SOC to SOM %
        'Clay_Percentage': clay_pct,
        'Silt_Percentage': silt_pct,
        'Cation_Exchange_Capacity_cmol_kg': cec_val,
        'Nitrogen_mg_kg': est_n_mg_kg,
        'Phosphorus_mg_kg': est_p_mg_kg,
        'Potassium_mg_kg': est_k_mg_kg,
        'Electrical_Conductivity_dS_m': 0.65
    }
    return profile

# Test with Thanjavur Farmland GPS
farm_lat, farm_lon = 10.7500, 79.1000
soil_data = fetch_complete_soil_profile(farm_lat, farm_lon)
print('=== Soil Profile Ingested ===')
for k, v in soil_data.items():
    print(f'  {k:<30}: {v}')

## 🌦️ 2. Weather Telemetry (Open-Meteo 72-Hour Forecast)
Fetches hourly and daily forecasts: temperature, precipitation sum, soil moisture, and wind speed.

In [ ]:
# Cell 5 | Open-Meteo Weather Gateway
def fetch_open_meteo_weather(lat, lon):
    """
    Query Open-Meteo API for 72-hour forecast.
    Returns aggregated meteorological metrics.
    """
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': lat,
        'longitude': lon,
        'hourly': ['temperature_2m', 'relative_humidity_2m', 'precipitation', 'soil_temperature_0cm', 'soil_moisture_0_to_1cm', 'wind_speed_10m'],
        'daily': ['precipitation_sum', 'temperature_2m_max', 'temperature_2m_min'],
        'forecast_days': 3,
        'timezone': 'auto'
    }
    
    try:
        res = requests.get(url, params=params, timeout=15)
        if res.status_code == 200:
            data = res.json()
            hourly = data.get('hourly', {})
            daily = data.get('daily', {})
            
            # Aggregate next 48h and 72h precipitation
            precip_hourly = hourly.get('precipitation', [0]*72)
            precip_48h = sum(precip_hourly[:48]) if len(precip_hourly) >= 48 else sum(precip_hourly)
            precip_72h = sum(precip_hourly[:72]) if len(precip_hourly) >= 72 else sum(precip_hourly)
            
            avg_temp = float(np.mean(hourly.get('temperature_2m', [28.0])))
            max_temp = float(np.max(hourly.get('temperature_2m', [32.0])))
            avg_rh = float(np.mean(hourly.get('relative_humidity_2m', [65.0])))
            avg_wind = float(np.mean(hourly.get('wind_speed_10m', [10.0])))
            avg_soil_temp = float(np.mean(hourly.get('soil_temperature_0cm', [26.0])))
            avg_soil_moist_pct = float(np.mean(hourly.get('soil_moisture_0_to_1cm', [0.25])) * 100.0)  # m3/m3 to %
            
            return {
                'Air_Temperature_C': round(avg_temp, 2),
                'Max_Temperature_C': round(max_temp, 2),
                'Relative_Humidity_Percent': round(avg_rh, 2),
                'Rainfall_mm': round(precip_72h, 2),
                'Precipitation_Next_48h_mm': round(precip_48h, 2),
                'Precipitation_Next_72h_mm': round(precip_72h, 2),
                'Soil_Temperature_C': round(avg_soil_temp, 2),
                'Soil_Moisture_Percent': round(avg_soil_moist_pct, 2),
                'Wind_Speed_kmph': round(avg_wind, 2),
                'Forecast_Timestamp': datetime.now().isoformat()
            }
    except Exception as e:
        print(f'Open-Meteo Query Error: {e}')
    
    # Fallback typical tropical agricultural conditions
    return {
        'Air_Temperature_C': 29.5,
        'Max_Temperature_C': 34.0,
        'Relative_Humidity_Percent': 68.0,
        'Rainfall_mm': 12.0,
        'Precipitation_Next_48h_mm': 8.5,
        'Precipitation_Next_72h_mm': 12.0,
        'Soil_Temperature_C': 27.2,
        'Soil_Moisture_Percent': 26.5,
        'Wind_Speed_kmph': 11.2,
        'Forecast_Timestamp': datetime.now().isoformat()
    }

weather_data = fetch_open_meteo_weather(farm_lat, farm_lon)
print('=== Weather Telemetry Ingested ===')
for k, v in weather_data.items():
    print(f'  {k:<30}: {v}')

## 🏗️ 3. Feature Assembly for ML Regressor
Combine soil data + weather telemetry + farmer crop selection into model-ready features.

In [ ]:
# Cell 6 | Feature Assembler
def assemble_inference_features(crop_name, target_yield_ton, growth_stage='Vegetative', irrigation_type='Drip', soil_profile=None, weather_data=None):
    # Load label encoders from Division 1
    encoders_path = BASE_DIR / 'dats' / 'label_encoders.pkl'
    with open(encoders_path, 'rb') as f:
        le_map = pd.read_pickle(f)
        
    # Safe categorical mapping with fallback to known classes
    def safe_encode(encoder, val):
        if val in encoder.classes_:
            return int(encoder.transform([val])[0])
        return 0
        
    crop_enc = safe_encode(le_map['Crop_Type'], crop_name)
    soil_enc = safe_encode(le_map['Soil_Type'], soil_profile.get('Soil_Texture', 'Loam'))
    growth_enc = safe_encode(le_map['Growth_Stage'], growth_stage)
    irrig_enc = safe_encode(le_map['Irrigation_Type'], irrigation_type)
    
    feature_dict = {
        'Crop_Type_enc': crop_enc,
        'Soil_Type_enc': soil_enc,
        'Growth_Stage_enc': growth_enc,
        'Irrigation_Type_enc': irrig_enc,
        'Nitrogen_mg_kg': soil_profile['Nitrogen_mg_kg'],
        'Phosphorus_mg_kg': soil_profile['Phosphorus_mg_kg'],
        'Potassium_mg_kg': soil_profile['Potassium_mg_kg'],
        'Soil_pH': soil_profile['Soil_pH'],
        'Soil_Moisture_Percent': weather_data['Soil_Moisture_Percent'],
        'Soil_Organic_Matter_Percent': soil_profile['Soil_Organic_Matter_Percent'],
        'Soil_Temperature_C': weather_data['Soil_Temperature_C'],
        'Electrical_Conductivity_dS_m': soil_profile['Electrical_Conductivity_dS_m'],
        'Air_Temperature_C': weather_data['Air_Temperature_C'],
        'Relative_Humidity_Percent': weather_data['Relative_Humidity_Percent'],
        'Rainfall_mm': weather_data['Rainfall_mm'],
        'Target_Yield_ton_per_Hectare': float(target_yield_ton),
        'Crop_Age_Days': 45,
        'Chlorophyll_Index': 42.0,
        'Crop_Stress_Index': 15.0
    }
    return pd.DataFrame([feature_dict])

feature_row = assemble_inference_features('Paddy', target_yield_ton=5.5, soil_profile=soil_data, weather_data=weather_data)
print('=== Assembled Inference Vector (Shape: {}) ==='.format(feature_row.shape))
print(feature_row.T)

## 💾 4. Export Telemetry Context

In [ ]:
# Cell 7 | Export JSON artifacts
with open(BASE_DIR / 'soil_baseline.json', 'w') as f:
    json.dump(soil_data, f, indent=2)

with open(BASE_DIR / 'weather_telemetry.json', 'w') as f:
    json.dump(weather_data, f, indent=2)

print('Saved soil_baseline.json & weather_telemetry.json')
print('Ready for Division 5 -> 05_deficiency_detector.ipynb')